# LAB-HW-05 — First PS/Linux Boot + UART Console

**One new thing today:** boot the KV260 processor system into Linux and prove that the runtime host exists independently of PL programming.

Prerequisite: LAB-HW-00~04.

**Project Trace:** RMD-012B · T-HW-005/T-HW-011

## 1. Development host is not the runtime host

<svg xmlns="http://www.w3.org/2000/svg" width="900" height="270" viewBox="0 0 900 270" role="img" aria-label="LAB-HW-05 development host UART and KV260 Linux boot path">
  <rect x="25" y="80" width="170" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="110" y="112" text-anchor="middle" font-size="15">development host</text>
  <text x="110" y="136" text-anchor="middle" font-size="12">download / flash / UART</text>
  <rect x="255" y="80" width="125" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="318" y="112" text-anchor="middle" font-size="15">J4 FTDI</text>
  <text x="318" y="136" text-anchor="middle" font-size="12">USB UART</text>
  <rect x="450" y="35" width="180" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="540" y="68" text-anchor="middle" font-size="15">K26 PS / Linux</text>
  <text x="540" y="92" text-anchor="middle" font-size="12">runtime host</text>
  <rect x="450" y="160" width="180" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="540" y="193" text-anchor="middle" font-size="15">J11 microSD</text>
  <text x="540" y="217" text-anchor="middle" font-size="12">Ubuntu Server image</text>
  <rect x="700" y="80" width="165" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="782" y="112" text-anchor="middle" font-size="15">shell evidence</text>
  <text x="782" y="136" text-anchor="middle" font-size="12">kernel / OS / board</text>
  <path d="M195 122 L255 122" stroke="#333" stroke-width="2"/><polygon points="255,122 245,117 245,127" fill="#333"/>
  <path d="M380 122 L450 82" stroke="#333" stroke-width="2"/><polygon points="450,82 438,83 443,92" fill="#333"/>
  <path d="M540 160 L540 115" stroke="#333" stroke-width="2"/><polygon points="540,115 535,125 545,125" fill="#333"/>
  <path d="M630 82 L700 122" stroke="#333" stroke-width="2"/><polygon points="700,122 688,112 686,123" fill="#333"/>
</svg>

The development host downloads/flashes the image and opens UART. The **runtime host** is Linux running on the K26 processor system (PS).

JTAG programming PL and Linux boot are different paths. Do not merge them mentally.

## 2. Stage-2 Ubuntu image

The course authoring candidate is:

`iot-limerick-kria-classic-server-2404-classic-24.04-x07-20250423.img.xz`

Distribution: **Ubuntu Server 24.04 LTS** for AMD Kria K26 starter kits.

The source identity is recorded in:

`boards/kv260/runtime/ubuntu24_image.json`

Important evidence boundary: the visible Canonical download index used during authoring did not publish an upstream SHA-256. The repository therefore does **not** invent one. Compute and retain the hash of the file you actually downloaded; formal image-hash PASS stays blocked until the course manifest freezes an expected hash.

## 3. Record the downloaded image identity

From the repository root on the development host:

```bash
python boards/kv260/runtime/hash_image.py \
  /path/to/iot-limerick-kria-classic-server-2404-classic-24.04-x07-20250423.img.xz
```

Expected authoring-stage status:

`STATUS=RECORDED_UNVERIFIED`

That is intentional. Save the filename, byte size, and computed SHA-256 in T-HW-011 evidence.

## 4. Flash the microSD card

Use a **16 GB UHS-1 or larger** microSD card.

For the beginner path, use **Raspberry Pi Imager**, which is the tool currently recommended by the AMD Kria Ubuntu guide:

1. Select the downloaded image.
2. Select the correct microSD card.
3. Write the image.
4. Wait for completion and safely eject the card.
5. Insert the card into KV260 **J11**.

Be careful about the target drive. Image flashing overwrites the selected device.

## 5. Open the UART console before power-on

Connect J4 to the development host with a data-capable USB cable.

UART settings:

- 115200 baud
- 8 data bits
- no parity
- 1 stop bit
- no hardware/software flow control

On Windows/macOS, the AMD guide identifies the second enumerated FTDI serial port as the UART. Linux enumeration can vary; identify the new FTDI serial devices after connecting J4 instead of memorizing one `ttyUSB` number.

Start terminal logging **before** applying board power so the boot transcript begins at power-on.

## 6. Power on and reach the shell

With J11 microSD and J4 UART already connected, apply 12 V / 3 A power at J12.

Current Ubuntu first-login credentials documented by AMD:

- username: `ubuntu`
- password: `ubuntu`

The first login requires a password change.

Do not install packages, load PL firmware, or debug AXI yet. Today the success condition is simply: **the PS/runtime host booted and you reached a shell.**

## 7. Collect machine-readable boot evidence

Run:

```bash
uname -a
cat /etc/os-release
cat /proc/device-tree/model; echo
sudo xmutil boardid
sudo xmutil bootfw_status
```

Or, after copying the repository helper to the runtime host:

```bash
bash boards/kv260/runtime/collect_boot_info.sh
```

Retain the command output together with the UART boot transcript.

## 8. Boot firmware boundary

AMD documents that current boot firmware is important for OS compatibility. In this Lab, **observe first** with `xmutil bootfw_status`.

If Linux does not boot because firmware is too old, use AMD's official boot-firmware update/recovery path as a troubleshooting branch and record what changed.

Do not turn firmware rewriting into an unrecorded “try this” step.

## 9. Clean shutdown

Before removing power:

```bash
sudo shutdown -h now
```

Wait for shutdown to complete, then remove power.

This protects the microSD filesystem and is part of the Lab evidence, not optional housekeeping.

## 10. Expected Evidence / Save Evidence

Record at least:

- image filename, size, locally computed SHA-256;
- flashing tool;
- UART device/COM port and 115200 8N1/no-flow settings;
- complete UART boot transcript;
- first shell reached;
- `uname -a`, `/etc/os-release`, device-tree model;
- `xmutil boardid` and `xmutil bootfw_status`;
- board/carrier revision, Git commit, date;
- clean shutdown observation.

Copy and fill `boards/kv260/evidence/manifest.example.json`.

Because the expected image SHA-256 is not yet frozen, do not label T-HW-005 as a final image-hash PASS.

## 11. If it does not work

Debug by layer:

1. no power LEDs → revisit J12/power;
2. power/heartbeat but no UART → J4 cable, FTDI driver, serial-port selection, 115200 8N1/no flow control;
3. UART shows boot firmware but no Linux from SD → image write, J11 card, or boot-firmware compatibility;
4. Linux reaches login but `xmutil` is missing → OS/platform image mismatch; retain evidence instead of improvising;
5. filesystem warnings after power removal → stop hard-powering the board; always use clean shutdown.

## 12. Human Check

Explain:

1. Which computer is the development host?
2. Which processor runs the runtime host?
3. Why does a successful JTAG PL program not prove Linux booted?
4. Why is the image filename not enough to identify an artifact?
5. Why do we record boot firmware but avoid casually updating it in the main path?
6. Why must the board be shut down before removing power?

## 13. Official basis

- AMD UG1089 — Software Getting Started / boot devices / interfaces
- AMD Kria KV260 Ubuntu 24.04 boot guide — SD-card setup, first boot, firmware update
- Canonical — Install Ubuntu on AMD, Kria K26 Ubuntu Server 24.04 LTS
- AMD/Xilinx xmutil — `boardid` and `bootfw_status`

Source URLs are recorded in the project documentation and `ubuntu24_image.json`.